# Replication Project: The Long-Run Risks Model

## Requirements

In [8]:
import pandas as pd
import numpy as np
import pandas_datareader.data as web
import statsmodels.api as sm
np.random.seed(42)

## Section 1: Bansal & Yaron Calibration

### Initialisation

We first initialize the two sets of parameters for both parametrizations.

In [9]:
params_BY = {
    'mu_c': 0.0015, 'rho_x': 0.979, 'phi_e': 0.044, 'mu_d': 0.0015, 'phi': 3.0,
    'varphi': 4.50, 'pi': 0.0, 'sigma': 0.0078, 'sigma_w': 0.0000023, 'nu': 0.987,
    'gamma': 10.0, 'psi': 1.5, 'delta': 0.9980
}

params_BKY = {
    'mu_c': 0.0015, 'rho_x': 0.975, 'phi_e': 0.038, 'mu_d': 0.0015, 'phi': 2.5,
    'varphi': 5.96, 'pi': 2.6, 'sigma': 0.0072, 'sigma_w': 0.0000028, 'nu': 0.999,
    'gamma': 10.0, 'psi': 1.5, 'delta': 0.9989
}

### Calibration

Following the derivations in the report,

In [10]:
def solve_bansal_yaron(p, tol=1e-8, max_iter=1000):
    alpha = 1 - p['gamma']
    rho = 1 - (1 / p['psi'])
    beta = p['delta']
    bar_sigma2 = p['sigma']**2

    # Utility function resolution
    b0, b1 = 0.001, beta
    for i in range(max_iter):
        px = b1 / (1 - b1 * p['rho_x'])
        psigma = (b1 * (alpha / 2) * (1 + px**2 * p['phi_e']**2)) / (1 - b1 * p['nu'])
        log_u = (b0 + b1 * (p['mu_c'] + psigma * (1 - p['nu']) * bar_sigma2 + (alpha / 2) * psigma**2 * p['sigma_w']**2)) / (1 - b1)
        log_mu = log_u + p['mu_c'] + psigma * bar_sigma2 + (alpha / 2) * ((1 + px**2 * p['phi_e']**2) * bar_sigma2 + psigma**2 * p['sigma_w']**2)

        exp_term = np.exp(rho * log_mu)
        b1_new = (beta * exp_term) / ((1 - beta) + beta * exp_term)
        b0_new = (1 / rho) * np.log((1 - beta) + beta * exp_term) - b1_new * log_mu

        if abs(b1_new - b1) < tol and abs(b0_new - b0) < tol:
            b1, b0 = b1_new, b0_new
            break
        b1, b0 = b1_new, b0_new

    # Pricing Kernel
    m = np.log(beta) + (rho - 1) * p['mu_c'] - (alpha * (alpha - rho) / 2) * psigma**2 * p['sigma_w']**2
    mx, msigma = rho - 1, - (alpha * (alpha - rho) / 2) * (1 + px**2 * p['phi_e']**2)
    lambda_eta, lambda_e, lambda_w = alpha - 1, (alpha - rho) * px * p['phi_e'], (alpha - rho) * psigma

    # Price-Dividend Ratio
    z_bar = 5.0
    for i in range(max_iter):
        k1 = np.exp(z_bar) / (1 + np.exp(z_bar))
        k0 = np.log(1 + np.exp(z_bar)) - k1 * z_bar
        qx = (mx + p['phi']) / (1 - k1 * p['rho_x'])
        qsigma = (msigma + 0.5 * ((lambda_eta + p['pi'])**2 + (lambda_e + k1 * qx * p['phi_e'])**2 + p['varphi']**2)) / (1 - k1 * p['nu'])
        q0 = (m + k0 + p['mu_d'] + k1 * qsigma * (1 - p['nu']) * bar_sigma2 + 0.5 * (lambda_w + k1 * qsigma)**2 * p['sigma_w']**2) / (1 - k1)

        z_bar_new = q0 + qsigma * bar_sigma2
        if abs(z_bar_new - z_bar) < tol:
            z_bar = z_bar_new
            break
        z_bar = z_bar_new

    # Risk-free rate
    E_rf = -m - msigma * bar_sigma2 - 0.5 * (lambda_eta**2 + lambda_e**2) * bar_sigma2 - 0.5 * lambda_w**2 * p['sigma_w']**2

    # Cross-covariance between M and R
    cov_m_r = lambda_eta * p['pi'] * bar_sigma2 \
            + lambda_e * (k1 * qx * p['phi_e']) * bar_sigma2 \
            + lambda_w * (k1 * qsigma) * p['sigma_w']**2

    # Variance of the log-return R
    var_r = (p['pi']**2 + (k1 * qx * p['phi_e'])**2 + p['varphi']**2) * bar_sigma2 \
          + (k1 * qsigma)**2 * p['sigma_w']**2

    # Equity Premium
    E_premium = -cov_m_r #- 0.5 * var_r

    return {
        'Annual Risk-Free Rate (%)': E_rf * 12 * 100,
        'Annual Equity Premium (%)': E_premium * 12 * 100,
        'E[log(P/D)]': z_bar,
        'q0': q0, 'qx': qx, 'qsigma': qsigma, 'k0': k0, 'k1': k1,
        'm': m, 'mx': mx, 'msigma': msigma,
        'lambda_eta': lambda_eta, 'lambda_e': lambda_e, 'lambda_w': lambda_w, 'b0': b0, 'b1': b1
    }

In [11]:
sol_BY = solve_bansal_yaron(params_BY)
sol_BKY = solve_bansal_yaron(params_BKY)

df_q1 = pd.DataFrame({'BY Calibration': sol_BY, 'BKY Calibration': sol_BKY}).T
print("Results")
print(df_q1[['Annual Risk-Free Rate (%)', 'Annual Equity Premium (%)', 'E[log(P/D)]']].round(4).to_string())

print("\nLinearization Constants (b0, b1)")
print(f"BY  : b0 = {sol_BY['b0']:.6e}, b1 = {sol_BY['b1']:.8f}")
print(f"BKY : b0 = {sol_BKY['b0']:.6e}, b1 = {sol_BKY['b1']:.8f}")

Results
                 Annual Risk-Free Rate (%)  Annual Equity Premium (%)  E[log(P/D)]
BY Calibration                      2.5812                     5.5363       5.4884
BKY Calibration                     1.2759                     6.4250       5.6353

Linearization Constants (b0, b1)
BY  : b0 = -2.495240e-06, b1 = 0.99805735
BKY : b0 = -4.107150e-05, b1 = 0.99872205


## Section 2: Predictability Analysis

### Data

Lacking direct access to the standard CRSP/BEA datasets, we relied on Robert Shiller's historical dataset to reconstruct the macroeconomic and financial variables.

In [12]:
def build_quarterly_dataset(shiller_filepath):
    # Loading and converting Shiller data to Quarterly
    shiller = pd.read_excel(shiller_filepath, sheet_name='Data', skiprows=7)
    shiller = shiller.dropna(subset=['Date'])
    shiller['Year'] = np.floor(shiller['Date']).astype(int)
    shiller['Month'] = np.round((shiller['Date'] - shiller['Year']) * 100).astype(int)
    shiller['Date_dt'] = pd.to_datetime(shiller['Year'].astype(str) + '-' + shiller['Month'].astype(str).str.zfill(2) + '-01')
    shiller.set_index('Date_dt', inplace=True)
    shiller = shiller[['P', 'D', 'CPI']].apply(pd.to_numeric, errors='coerce')

    # Quarterly conversion (taking the last value of the quarter)
    shiller_q = shiller.resample('Q').last()
    shiller_q['D_Q'] = shiller_q['D'] / 4  # Shiller provides annualized dividends
    shiller_q.index = shiller_q.index.to_period('Q')

    # Downloading Consumption and T-Bills data (FRED)
    fred_data = web.DataReader(['A794RX0Q048SBEA', 'TB3MS'], 'fred', '1947-01-01', '2008-12-31')
    fred_q = fred_data.resample('Q').last()
    fred_q.rename(columns={'A794RX0Q048SBEA': 'Cons_Real_PerCapita', 'TB3MS': 'rf_nom_ann'}, inplace=True)
    fred_q.index = fred_q.index.to_period('Q')

    df = shiller_q.join(fred_q, how='inner')

    # BY variables
    df['dc'] = np.log(df['Cons_Real_PerCapita'] / df['Cons_Real_PerCapita'].shift(1))
    df['pd'] = np.log(df['P'] / df['D'])  # annualized D

    df['Real_D_Q'] = df['D_Q'] / df['CPI']
    df['dd'] = np.log(df['Real_D_Q'] / df['Real_D_Q'].shift(1))

    df['Real_P'] = df['P'] / df['CPI']
    df['rm'] = np.log((df['Real_P'] + df['Real_D_Q']) / df['Real_P'].shift(1))

    # Constructing the proxy for the ex-ante risk-free rate
    df['rf_nom_q'] = df['rf_nom_ann'] / 100 / 4
    df['inflation_q'] = np.log(df['CPI'] / df['CPI'].shift(1))
    df['rf_ex_post'] = df['rf_nom_q'] - df['inflation_q']
    df['inflation_1y'] = np.log(df['CPI'] / df['CPI'].shift(4))
    # The target is the ex-post rf of the FOLLOWING quarter
    df['rf_ex_post_target'] = df['rf_nom_q'] - df['inflation_q'].shift(-1)

    temp = df.dropna(subset=['rf_ex_post_target', 'rf_nom_q', 'inflation_1y'])
    X = sm.add_constant(temp[['rf_nom_q', 'inflation_1y']])
    y = temp['rf_ex_post_target']
    model = sm.OLS(y, X).fit()

    df['rf'] = model.predict(sm.add_constant(df[['rf_nom_q', 'inflation_1y']]))

    df = df.dropna(subset=['dc', 'dd', 'pd', 'rm', 'rf'])

    print(f"Quarterly dataset: {df.shape[0]} quarters (from {df.index.min()} to {df.index.max()})")
    return df

In [13]:
def build_annual_dataset(shiller_filepath):
    # Loading and converting Shiller data to Annual
    shiller = pd.read_excel(shiller_filepath, sheet_name='Data', skiprows=7)
    shiller = shiller.dropna(subset=['Date'])
    shiller['Year'] = np.floor(shiller['Date']).astype(int)
    shiller['Month'] = np.round((shiller['Date'] - shiller['Year']) * 100).astype(int)
    shiller['Date_dt'] = pd.to_datetime(shiller['Year'].astype(str) + '-' + shiller['Month'].astype(str).str.zfill(2) + '-01')
    shiller.set_index('Date_dt', inplace=True)
    shiller = shiller[['P', 'D', 'CPI']].apply(pd.to_numeric, errors='coerce')

    # Annual conversion (taking the last value of the year)
    shiller_a = shiller.resample('Y').last()
    shiller_a['D_A'] = shiller_a['D']  # Shiller already provides annualized dividends
    shiller_a.index = shiller_a.index.to_period('Y')

    # Downloading Consumption and T-Bills data (FRED)
    fred_data = web.DataReader(['A794RX0Q048SBEA', 'TB3MS'], 'fred', '1947-01-01', '2008-12-31')
    fred_a = fred_data.resample('Y').last()
    fred_a.rename(columns={'A794RX0Q048SBEA': 'Cons_Real_PerCapita', 'TB3MS': 'rf_nom_ann'}, inplace=True)
    fred_a.index = fred_a.index.to_period('Y')


    df = shiller_a.join(fred_a, how='inner')

    # BY Variables
    df['dc'] = np.log(df['Cons_Real_PerCapita'] / df['Cons_Real_PerCapita'].shift(1))
    df['pd'] = np.log(df['P'] / df['D'])

    df['Real_D_A'] = df['D_A'] / df['CPI']
    df['dd'] = np.log(df['Real_D_A'] / df['Real_D_A'].shift(1))

    df['Real_P'] = df['P'] / df['CPI']
    df['rm'] = np.log((df['Real_P'] + df['Real_D_A']) / df['Real_P'].shift(1))

    # Constructing the proxy for the ex-ante risk-free rate (Annual)
    df['rf_nom_a'] = df['rf_nom_ann'] / 100
    df['inflation_a'] = np.log(df['CPI'] / df['CPI'].shift(1))
    df['rf_ex_post'] = df['rf_nom_a'] - df['inflation_a']

    df['inflation_1y'] = df['inflation_a']
    df['rf_ex_post_target'] = df['rf_nom_a'] - df['inflation_a'].shift(-1)

    temp = df.dropna(subset=['rf_ex_post_target', 'rf_nom_a', 'inflation_1y'])
    X = sm.add_constant(temp[['rf_nom_a', 'inflation_1y']])
    y = temp['rf_ex_post_target']
    model = sm.OLS(y, X).fit()

    df['rf'] = model.predict(sm.add_constant(df[['rf_nom_a', 'inflation_1y']]))

    df = df.dropna(subset=['dc', 'dd', 'pd', 'rm', 'rf'])

    print(f"Annual dataset: {df.shape[0]} years (from {df.index.min()} to {df.index.max()})")
    return df

In [14]:
# Quarterly
df_quarterly = build_quarterly_dataset('ie_data.xls')
print(df_quarterly[['dc', 'dd', 'pd', 'rm', 'rf']].head())

/tmp/ipykernel_43275/2609379059.py:12: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  shiller_q = shiller.resample('Q').last()


Quarterly dataset: 244 quarters (from 1948Q1 to 2008Q4)
              dc        dd        pd        rm        rf
1948Q1  0.000947  0.011834  2.822778 -0.035038 -0.008224
1948Q2  0.007542 -0.029476  2.985088  0.145388 -0.011915
1948Q3 -0.002822  0.006796  2.896737 -0.067848 -0.007608
1948Q4  0.003343  0.083153  2.793208 -0.005186 -0.002490
1949Q1 -0.002820  0.064894  2.722235  0.010219 -0.000617


/tmp/ipykernel_43275/2609379059.py:18: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  fred_q = fred_data.resample('Q').last()


In [15]:
# Annual
df_annual = build_annual_dataset('ie_data.xls')
print(df_annual[['dc', 'dd', 'pd', 'rm', 'rf']].head())

/tmp/ipykernel_43275/1477241278.py:12: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  shiller_a = shiller.resample('Y').last()


Annual dataset: 61 years (from 1948 to 2008)
            dc        dd        pd        rm        rf
1948  0.009011  0.072307  2.793208  0.040537 -0.011426
1949  0.016449  0.224564  2.674753  0.172762  0.012752
1950  0.035876  0.196605  2.597891  0.191533 -0.023837
1951 -0.004862 -0.099942  2.809574  0.170227 -0.021263
1952  0.044266 -0.007519  2.916044  0.151684  0.006174


/tmp/ipykernel_43275/1477241278.py:18: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  fred_a = fred_data.resample('Y').last()


### Estimation

In [16]:
def simulate_and_build_tables(p, sol, n_sims=100000, n_periods=247, freq='Q', subset_size=2000):
    # subset for speed — 2000 paths sufficient

    # Configuration based on frequency
    if freq == 'Q':
        months_per_period = 3
        horizons = {'1Y': 4, '3Y': 12, '5Y': 20}
        m_mean, m_std = 400, 200 # Multipliers to annualize from quarterly
    else: # 'A' for Annual
        months_per_period = 12
        horizons = {'1Y': 1, '3Y': 3, '5Y': 5}
        m_mean, m_std = 100, 100

    n_months = n_periods * months_per_period
    burn_in = 120
    total_months = n_months + burn_in

    x = np.zeros(n_sims)
    sigma2 = np.ones(n_sims) * (p['sigma']**2)

    mc_dc, mc_dd, mc_rm, mc_rf, mc_pd = (np.zeros((n_months, n_sims)) for _ in range(5))

    for t in range(total_months):
        eta, e, w, u = np.random.randn(4, n_sims)
        sigma2_censored = np.maximum(sigma2, 1e-10)
        sigma_t = np.sqrt(sigma2_censored)

        dc_t = p['mu_c'] + x + sigma_t * eta
        dd_t = p['mu_d'] + p['phi'] * x + p['varphi'] * sigma_t * u + p['pi'] * sigma_t * eta
        rf_t = -sol['m'] - sol['mx'] * x - sol['msigma'] * sigma2_censored \
               - 0.5 * (sol['lambda_eta']**2 + sol['lambda_e']**2) * sigma2_censored \
               - 0.5 * sol['lambda_w']**2 * (p['sigma_w']**2)
        z_t = sol['q0'] + sol['qx'] * x + sol['qsigma'] * sigma2_censored

        x_next = p['rho_x'] * x + p['phi_e'] * sigma_t * e
        sigma2_next = p['sigma']**2 + p['nu'] * (sigma2 - p['sigma']**2) + p['sigma_w'] * w
        sigma2_next_censored = np.maximum(sigma2_next, 1e-10)
        z_next = sol['q0'] + sol['qx'] * x_next + sol['qsigma'] * sigma2_next_censored

        rm_t = sol['k0'] + sol['k1'] * z_next - z_t + dd_t

        if t >= burn_in:
            idx = t - burn_in
            mc_dc[idx], mc_dd[idx], mc_rf[idx], mc_rm[idx], mc_pd[idx] = dc_t, dd_t, rf_t, rm_t, z_t

        x, sigma2 = x_next, sigma2_next

    niveau_c = np.exp(np.cumsum(mc_dc, axis=0))
    niveau_d = np.exp(np.cumsum(mc_dd, axis=0))

    # Aggregation based on months per period
    c_q_sum = niveau_c.reshape(n_periods, months_per_period, n_sims).sum(axis=1)
    d_q_sum = niveau_d.reshape(n_periods, months_per_period, n_sims).sum(axis=1)

    dc_q = np.log(c_q_sum[1:] / c_q_sum[:-1])
    dd_q = np.log(d_q_sum[1:] / d_q_sum[:-1])

    rf_q = mc_rf.reshape(n_periods, months_per_period, n_sims).sum(axis=1)
    rm_q = mc_rm.reshape(n_periods, months_per_period, n_sims).sum(axis=1)
    ex_rm_q = rm_q[1:] - rf_q[1:]

    pd_q = np.zeros((n_periods, n_sims))
    period_ends = np.arange(months_per_period - 1, n_months, months_per_period)

    for q_idx, t_month in enumerate(period_ends):
        if t_month >= 11:
            d_12m = np.sum(niveau_d[t_month-11 : t_month+1, :], axis=0)
        else:
            d_12m = np.sum(niveau_d[0 : t_month+1, :], axis=0) * (12 / (t_month + 1))

        # P/D = P_t / Sum_12m(D)
        pd_q[q_idx, :] = mc_pd[t_month, :] + np.log(niveau_d[t_month, :]) - np.log(d_12m)

    pd_q_aligned = pd_q[:-1, :]
    past_ex_rm_q = rm_q[:-1] - rf_q[:-1]

    # Tables
    res = {}

    for name, var_data in [('dc', dc_q), ('dd', dd_q), ('rm_rf', ex_rm_q), ('rf', rf_q[1:]), ('pd', pd_q_aligned)]:
        mult_mean = 1 if name == 'pd' else m_mean
        mult_std = 1 if name == 'pd' else m_std

        path_means = np.mean(var_data, axis=0)
        res[f'Mean({name})'] = np.median(path_means) * mult_mean

        path_stds = np.std(var_data, axis=0)
        res[f'Std({name})']  = np.median(path_stds) * mult_std

        covs = np.mean(var_data[:-1, :] * var_data[1:, :], axis=0) - np.mean(var_data[:-1, :], axis=0) * np.mean(var_data[1:, :], axis=0)
        vars = np.std(var_data[:-1, :], axis=0) * np.std(var_data[1:, :], axis=0)
        ar1_array = covs / vars
        res[f'AR1({name})'] = np.nanmedian(ar1_array)

    targets_B = ['rm_rf', 'dc', 'dd']
    targets_C = ['rm_rf', 'dc', 'dd', 'rf']

    r2_b = {h: {t: [] for t in targets_B} for h in horizons}
    r2_c = {h: {t: [] for t in targets_C} for h in horizons}

    for s in range(subset_size):
        pd_s = pd_q_aligned[:, s]
        rm_rf_s = ex_rm_q[:, s]       # Target (t to t+1)
        past_rm_rf_s = past_ex_rm_q[:, s] # Predictor (t-1 to t)
        rf_s = rf_q[1:, s]            # Known at t for period t to t+1
        dc_s = dc_q[:, s]
        dd_s = dd_q[:, s]

        for h_name, J in horizons.items():
            if len(rm_rf_s) <= J: continue

            # Aggregation of future returns over horizon J
            Y = {
                'rm_rf': np.array([np.sum(rm_rf_s[t:t+J]) for t in range(len(rm_rf_s)-J)]),
                'dc': np.array([np.sum(dc_s[t:t+J]) for t in range(len(dc_s)-J)]),
                'dd': np.array([np.sum(dd_s[t:t+J]) for t in range(len(dd_s)-J)]),
                'rf': np.array([np.sum(rf_s[t+1:t+1+J]) for t in range(len(rf_s)-J)])
            }

            X_pd = sm.add_constant(pd_s[:len(Y['rm_rf'])])
            X_multi = sm.add_constant(np.column_stack((past_rm_rf_s[:len(Y['rm_rf'])], pd_s[:len(Y['rm_rf'])], rf_s[:len(Y['rm_rf'])])))

            for target in targets_B:
                r2_b[h_name][target].append(sm.OLS(Y[target], X_pd).fit().rsquared)
            for target in targets_C:
                r2_c[h_name][target].append(sm.OLS(Y[target], X_multi).fit().rsquared)

    for h in horizons:
        for t in targets_B:
            res[f'Univariate R2: {t} ({h})'] = np.median(r2_b[h][t])
        for t in targets_C:
            res[f'Multivariate R2: {t} ({h})'] = np.median(r2_c[h][t])

    return res

In [17]:
# Quarterly
res_BY_Q = simulate_and_build_tables(params_BY, sol_BY, n_periods=247, freq='Q')
res_BKY_Q = simulate_and_build_tables(params_BKY, sol_BKY, n_periods=247, freq='Q')
df_sim_Q = pd.DataFrame({'BY (Quarterly)': res_BY_Q, 'BKY (Quarterly)': res_BKY_Q})
print(df_sim_Q.round(3))

                             BY (Quarterly)  BKY (Quarterly)
Mean(dc)                              1.800            1.800
Std(dc)                               2.434            2.189
AR1(dc)                               0.312            0.277
Mean(dd)                              1.794            1.808
Std(dd)                              10.512           13.622
AR1(dd)                               0.258            0.217
Mean(rm_rf)                           4.158            4.533
Std(rm_rf)                           16.453           17.913
AR1(rm_rf)                           -0.004           -0.005
Mean(rf)                              2.586            1.315
Std(rf)                               0.613            0.468
AR1(rf)                               0.948            0.944
Mean(pd)                              3.011            3.158
Std(pd)                               0.180            0.170
AR1(pd)                               0.897            0.873
Univariate R2: rm_rf (1Y

In [18]:
# Annual
res_BY_A = simulate_and_build_tables(params_BY, sol_BY, n_periods=79, freq='A') # 79 to match the original time range
res_BKY_A = simulate_and_build_tables(params_BKY, sol_BKY, n_periods=79, freq='A')
df_sim_A = pd.DataFrame({'BY (Annual)': res_BY_A, 'BKY (Annual)': res_BKY_A})
print(df_sim_A.round(3))

                             BY (Annual)  BKY (Annual)
Mean(dc)                           1.800         1.798
Std(dc)                            2.797         2.383
AR1(dc)                            0.475         0.405
Mean(dd)                           1.790         1.797
Std(dd)                           11.120        13.436
AR1(dd)                            0.370         0.263
Mean(rm_rf)                        4.153         4.556
Std(rm_rf)                        16.321        17.782
AR1(rm_rf)                        -0.014        -0.012
Mean(rf)                           2.583         1.298
Std(rf)                            1.206         0.930
AR1(rf)                            0.821         0.814
Mean(pd)                           3.011         3.154
Std(pd)                            0.184         0.179
AR1(pd)                            0.663         0.663
Univariate R2: rm_rf (1Y)          0.007         0.013
Univariate R2: dc (1Y)             0.315         0.140
Univariate

In [19]:
def compute_empirical_tables(df, freq='Q'):
    """
    Computes unconditional moments and predictive R2 for a historical DataFrame.
    freq = 'Q' (Quarterly) or 'A' (Annual)
    Needs: 'dc', 'dd', 'rm', 'rf', 'pd'
    """
    df = df.copy().dropna()
    df['rm_rf'] = df['rm'] - df['rf'].shift(1)
    df = df.dropna()

    # Horizons
    if freq == 'Q':
        mult_mean, mult_std = 4, 2
        horizons = {'1Y': 4, '3Y': 12, '5Y': 20}
    else: # Annual
        mult_mean, mult_std = 1, 1
        horizons = {'1Y': 1, '3Y': 3, '5Y': 5}

    res = {}

    # Unconditional Moments
    for var in ['dc', 'dd', 'rm_rf', 'rf', 'pd']:
        m_m = 1 if var == 'pd' else mult_mean
        m_s = 1 if var == 'pd' else mult_std

        mean_val = df[var].mean() * m_m * (1 if var == 'pd' else 100)
        std_val = df[var].std() * m_s * (1 if var == 'pd' else 100)
        ar1_val = df[var].autocorr(lag=1)

        suffix = "" if var == 'pd' else " (%)"
        res[f'Mean({var}){suffix}'] = mean_val
        res[f'Std({var}){suffix}'] = std_val
        res[f'AR1({var})'] = ar1_val

    # Predictive Regressions (Univariate and Multivariate)
    targets_B = ['rm_rf', 'dc', 'dd']
    targets_C = ['rm_rf', 'dc', 'dd', 'rf']

    for h_name, J in horizons.items():
        if len(df) <= J: continue
        Y_sums = {var: [] for var in set(targets_B + targets_C)}
        X_pd, X_rm_rf, X_rf = [], [], []

        for i in range(len(df) - J):
            X_pd.append(df['pd'].iloc[i])
            X_rm_rf.append(df['rm_rf'].iloc[i])
            X_rf.append(df['rf'].iloc[i])


            for var in Y_sums.keys():
                Y_sums[var].append(df[var].iloc[i+1 : i+1+J].sum())

        X_pd = sm.add_constant(np.array(X_pd))
        X_multi = sm.add_constant(np.column_stack((X_rm_rf, X_pd[:, 1], X_rf)))

        # Univariate regressions
        for var in targets_B:
            model_B = sm.OLS(Y_sums[var], X_pd).fit()
            res[f'Univariate R2: {var} ({h_name})'] = model_B.rsquared

        # Multivariate regressions
        for var in targets_C:
            model_C = sm.OLS(Y_sums[var], X_multi).fit()
            res[f'Multivariate R2: {var} ({h_name})'] = model_C.rsquared

    return pd.Series(res)

In [20]:
# Quarterly
emp_results_Q = compute_empirical_tables(df_quarterly, freq='Q')

print("EMPIRICAL RESULTS (Quarterly)")
print(emp_results_Q.round(3).to_string())

EMPIRICAL RESULTS (Quarterly)
Mean(dc) (%)                    2.242
Std(dc) (%)                     1.693
AR1(dc)                         0.040
Mean(dd) (%)                    2.131
Std(dd) (%)                     4.051
AR1(dd)                         0.414
Mean(rm_rf) (%)                 5.987
Std(rm_rf) (%)                 14.474
AR1(rm_rf)                      0.121
Mean(rf) (%)                    1.158
Std(rf) (%)                     0.784
AR1(rf)                         0.862
Mean(pd)                        3.439
Std(pd)                         0.436
AR1(pd)                         0.986
Univariate R2: rm_rf (1Y)       0.088
Univariate R2: dc (1Y)          0.006
Univariate R2: dd (1Y)          0.001
Multivariate R2: rm_rf (1Y)     0.093
Multivariate R2: dc (1Y)        0.086
Multivariate R2: dd (1Y)        0.040
Multivariate R2: rf (1Y)        0.573
Univariate R2: rm_rf (3Y)       0.184
Univariate R2: dc (3Y)          0.013
Univariate R2: dd (3Y)          0.000
Multivariate R2: rm_

In [21]:
# Annual
emp_results_A = compute_empirical_tables(df_annual, freq='A')

print("EMPIRICAL RESULTS (Annually)")
print(emp_results_A.round(3).to_string())

EMPIRICAL RESULTS (Annually)
Mean(dc) (%)                    2.308
Std(dc) (%)                     1.891
AR1(dc)                         0.130
Mean(dd) (%)                    2.085
Std(dd) (%)                     5.618
AR1(dd)                         0.506
Mean(rm_rf) (%)                 6.233
Std(rm_rf) (%)                 16.006
AR1(rm_rf)                      0.056
Mean(rf) (%)                    1.286
Std(rf) (%)                     1.409
AR1(rf)                         0.662
Mean(pd)                        3.446
Std(pd)                         0.439
AR1(pd)                         0.936
Univariate R2: rm_rf (1Y)       0.068
Univariate R2: dc (1Y)          0.006
Univariate R2: dd (1Y)          0.007
Multivariate R2: rm_rf (1Y)     0.111
Multivariate R2: dc (1Y)        0.069
Multivariate R2: dd (1Y)        0.137
Multivariate R2: rf (1Y)        0.448
Univariate R2: rm_rf (3Y)       0.137
Univariate R2: dc (3Y)          0.010
Univariate R2: dd (3Y)          0.006
Multivariate R2: rm_r